# Topic 5: Provenance & Debugging

RAG's whole promise is *grounded answers* -- the LLM only speaks from your documents.
But in Topic 4 we watched it say *"Paranoid" by Black Sabbath*, which was **not** in the
chunks. So this topic answers the question:

> **"Which chunks did this answer actually come from?"**

If we can *see* the source chunks for every answer, we can:
- catch the model leaking outside the context,
- catch weak / duplicate retrieval (the Topic 1 bug),
- fix the retrieval at the source instead of patching the prompt.

## The 3 tools

1. **Tool 1 -- Inspect the chunks.** Look at what `retriever.invoke()` returns *before* it reaches the LLM.
2. **Tool 2 -- Fact-check the answer against its chunks.** Flag any answer content that is NOT supported by the context (a *leak*).
3. **Tool 3 -- Fix at the source.** Fix what you find (e.g. deduplicate the repeated-verse chokehold from Topic 1).

> Note: LLMs are **stochastic** -- the same question can yield a clean answer one run and a leaking one the next.
> That is exactly why provenance must be a *routine check*, not a one-off.

## Tool 1 -- Inspect what the retriever hands the LLM

Before the LLM ever sees text, the retriever picks the grounding chunks. Read them
directly from `retriever.invoke(query)` -- the `Document`s carry both `page_content`
(the text) and `metadata` (source file, offsets, etc.).

## Tool 2 -- Fact-check the answer against its chunks

Run the full chain, then scan the answer for tokens/claims that exist in the answer but
**not** in the retrieved context. Those are leaks -- facts the model imported from its
own memory rather than from your documents.

```python
for token in interesting_tokens:
    in_context = token.lower() in context.lower()
    in_answer  = token.lower() in answer.lower()
    flag       = "OK" if (not in_answer) or in_context else "*** LEAK ***"
```

## Tool 3 -- Fix at the source (dedup)

The Topic 1 bug: a lyric verse repeats verbatim in the album, so two near-identical
chunks both rank in the top-k and get served to the LLM, wasting context and biasing
the answer. Fix **in the transformation layer**, not the prompt: dedupe on content as
you build the `{context}` block.

```python
def format_docs_dedup(docs):
    seen, kept = set(), []
    for d in docs:
        text = d.page_content.strip()          # normalize whitespace
        if text not in seen:                    # keep first, drop repeats
            seen.add(text)
            kept.append(text)
    return "\n\n".join(kept)
```

Verified on eve: `k=4` returned 2 copies of the repeated verse -> dedup delivers
**3 unique chunks** (measured: `out.count("\\n\\n") + 1`).

## Topic 5 complete -- the summary

- **Trust nothing.** Inspect the chunks, verify the answer against them, fix retrieval at the source.
- Tool 1: `retriever.invoke()` before the LLM shows the grounding `Document`s (text + metadata).
- Tool 2: token scan answer-vs-context flags *leaks* ("Black Sabbath" was a classic one).
- Tool 3: dedup in `format_docs`, not in the prompt -- cheap, deterministic, and it fixes the Topic 1 repeated-verse bug (4 -> 3 unique chunks).

In [ ]:
# ---- Setup shared by the cells below ----
# Same FAISS store from Step 4, rebuilt so the notebook is self-contained.
from pathlib import Path

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# robust path finding: works no matter which folder Jupyter opened the notebook in
def find_album():
    roots = [Path.cwd(), *Path.cwd().parents]
    for root in roots:
        for cand in (
            root / "data_ingestion" / "heylog_eve_album.txt",
            root / "section_5_LangChain" / "data_ingestion" / "heylog_eve_album.txt",
        ):
            if cand.exists():
                return cand
    raise FileNotFoundError("heylog_eve_album.txt not found")

emb = OllamaEmbeddings(model="nomic-embed-text")
split = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
chunks = split.split_documents(TextLoader(str(find_album())).load())
store = FAISS.from_documents(chunks, emb)
retriever = store.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

prompt = PromptTemplate.from_template(
    "Answer using ONLY the context. If absent, say \"I don't know.\"\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\nAnswer:"
)
llm = ChatOllama(model="llama3:8b")
parser = StrOutputParser()

print("store ready:", type(store).__name__, "| chunks:", store.index.ntotal)

In [ ]:
# ---- Tool 1: inspect the chunks BEFORE they reach the LLM ----
query = "what is the song paranoid about?"
print("=== chunks that WILL ground the answer ===")
for i, d in enumerate(retriever.invoke(query)):
    print(f"[chunk {i}] source={Path(d.metadata['source']).name}")
    print("    ", d.page_content[:70].replace("\n", " "))

In [ ]:
# ---- Tool 2: fact-check the answer against its chunks ----
# Full chain first (the "raw" / no-provenance version).
chain_raw = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | parser
)
answer = chain_raw.invoke(query)
context = format_docs(retriever.invoke(query))

print("ANSWER:", answer[:220])
print("\n=== leak check (tokens in answer but NOT in retrieved chunks) ===")
for token in ["Black Sabbath", "Adam and Eve", "Eve", "paranoia", "deer"]:
    in_context = token.lower() in context.lower()
    in_answer = token.lower() in answer.lower()
    flag = "OK" if (not in_answer) or in_context else "*** LEAK ***"
    print(f"  {token:<16} in-context={in_context!s:<5} in-answer={in_answer!s:<5} -> {flag}")

# NOTE: rerun this cell a few times -- answers are stochastic, so the leak
# check matters every single time, not just once.

In [ ]:
# ---- Tool 3: dedup at the source (fixes the Topic 1 repeated-verse bug) ----
k4 = store.as_retriever(search_kwargs={"k": 4})
query_dup = "shame like Adam and Eve"

def format_docs_dedup(docs):
    seen, kept = set(), []
    for d in docs:
        text = d.page_content.strip()
        if text not in seen:
            seen.add(text)
            kept.append(text)
    return "\n\n".join(kept)

raw = k4.invoke(query_dup)
print(f"retrieved: {len(raw)} chunks")
for i, d in enumerate(raw):
    print(f"  [raw {i}] {d.page_content[:44].replace(chr(10), ' ')}")

out = format_docs_dedup(raw)
unique = out.count("\n\n") + 1
print(f"after dedup: {unique} unique contexts delivered to the LLM")

## PART B -- The same 3 tools, on a different corpus (AI blogs)

The three tools are *skill*, not keystrokes. This section re-runs Tool 1/2/3 on a
completely different, much larger source: **`ai_blog_rag_articles.txt`** (24 KB,
295 chunks) scraped from three real AI/RAG blog posts (Thoughtworks, Microsoft,
StackAI). Same tools, new data -- proves the method transfers.

### Tool 1 (AI blogs) -- inspect the chunks first

Same move as before: look at the `Document`s BEFORE any LLM sees them. Query
"what technique combines keyword search and vector search?" should land on the
"hybrid search" section -- and only that section.

### Tool 2 (AI blogs) -- fact-check the answer against its chunks

Same leak scan. Note the answer for "what is corrective RAG?" comes back weak
("a feature of corrective RAG") because at chunk_size=120 the CRAG definition is
fragmented across chunk boundaries. Provenance reveals *why* the answer is weak
before you blame the model -- the fix is a chunking decision (Topic 8), not a
prompt tweak.

### Tool 3 (AI blogs) -- dedup, but measure first

The clean multi-source corpus has no repeated-verse problem, so dedup found few
duplicates. Lesson: **inspect the data first, then apply the fix that is actually
needed.** The dedup code itself is proven (it is identical to the eve fix).

## Topic 5 -- the real takeaway

- The 3 tools are a **transferable workflow**: inspect chunks -> verify the answer
  against them -> fix at the source.
- On the album, the fix was dedup (repeated verse). On the blogs, the weaknesses
  are chunk fragmentation and breadth -- the *same* workflow exposed different
  problems in different data. That is the whole point of provenance.

In [ ]:
# ---- PART B setup: build the store from the AI blogs corpus ----
# 295 chunks from 24 KB of scraped AI blog text. Reuse the same pattern.
from pathlib import Path

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

def find_corpus(fname):
    roots = [Path.cwd(), *Path.cwd().parents]
    for root in roots:
        for cand in (root / "data_ingestion" / fname,
                     root / "section_5_LangChain" / "data_ingestion" / fname):
            if cand.exists():
                return cand
    raise FileNotFoundError(f"{fname} not found")

emb = OllamaEmbeddings(model="nomic-embed-text")
split = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
chunks = split.split_documents(TextLoader(str(find_corpus("ai_blog_rag_articles.txt"))).load())
store = FAISS.from_documents(chunks, emb)
retriever = store.as_retriever(search_kwargs={"k": 3})
print("PART B store:", type(store).__name__, "| chunks:", store.index.ntotal)

In [ ]:
# ---- TOOL 1 (AI blogs): inspect chunks BEFORE the LLM ----
query = "what technique combines keyword search and vector search?"
print("=== Tool 1: chunks that will ground the answer ===")
for i, d in enumerate(retriever.invoke(query)):
    print(f"[chunk {i}] source={Path(d.metadata['source']).name}")
    print("           ", d.page_content[:75].replace("\n", " "))

In [ ]:
# ---- TOOL 2 (AI blogs): full chain + leak scan ----
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

prompt = PromptTemplate.from_template(
    "Answer using ONLY the context. If absent, say \"I don't know.\"\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\nAnswer:"
)
llm = ChatOllama(model="llama3:8b")
parser = StrOutputParser()

chain = ({"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm | parser)
query = "what is corrective RAG?"
answer = chain.invoke(query)
context = format_docs(retriever.invoke(query))

print("Q:", query)
print("ANSWER:", answer[:280])
print("\n=== leak check ===")
for token in ["Corrective RAG", "self-reflection", "evaluation", "ChromaDB", "FAISS", "OpenAI"]:
    in_context = token.lower() in context.lower()
    in_answer = token.lower() in answer.lower()
    flag = "OK" if (not in_answer) or in_context else "*** LEAK ***"
    print(f"  {token:<16} in-context={in_context!s:<5} in-answer={in_answer!s:<5} -> {flag}")

In [ ]:
# ---- TOOL 3 (AI blogs): dedup, but measure first ----
# Clean multi-source corpus -> few duplicates. The method is proven; here we
# inspect first and confirm the cleanup is barely needed.
retriever5 = store.as_retriever(search_kwargs={"k": 5})

def format_docs_dedup(docs):
    seen, kept = set(), []
    for d in docs:
        text = d.page_content.strip()
        if text not in seen:
            seen.add(text)
            kept.append(text)
    return "\n\n".join(kept)

for query in ["What are the limitations of RAG-fusion?",
              "reciprocal rank fusion",
              "what is vector search?"]:
    raw = retriever5.invoke(query)
    out = format_docs_dedup(raw)
    unique = out.count("\n\n") + 1
    print(f"Q: {query}  retrieved={len(raw)} -> after dedup={unique}")